# CNU Campus ChatBot — 제출 노트북 (Gemma 3 / torch 2.5.1)

**과제 명세 환경 torch 2.5.1 고정.** 설치 때문에 런타임 재시작이 2번 필요해서
완전 원클릭은 아니야. 아래 순서대로 실행:

1. **[1단계]** torch 2.5.1 설치 → 자동 재시작
2. **[2단계]** transformers 4.50 + 라이브러리 설치 → 자동 재시작
3. **[3단계]~[8단계]** 재시작 후 위에서부터 끝까지 실행 (재시작 없음)

## 최초 1회만
- gemma-3 라이선스 수락: https://huggingface.co/google/gemma-3-12b-it → Agree
- 런타임 유형 = **T4 GPU**

## [1단계] torch 2.5.1 설치 → 자동 재시작

In [ ]:
import os
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip uninstall -y -q torchcodec 2>/dev/null
print("torch 2.5.1 설치 완료 — 재시작합니다. 재시작되면 [2단계] 셀부터.")
os.kill(os.getpid(), 9)

## [2단계] (재시작 후) transformers 4.50 + 라이브러리 → 자동 재시작
torch를 2.5.1로 핀에 박아 pip가 다시 올리지 못하게 한다.

In [ ]:
import torch, os
assert torch.__version__.startswith("2.5.1"), "torch가 2.5.1이 아님 — [1단계]부터 다시"
!pip install -q transformers==4.50.0 accelerate "bitsandbytes>=0.45" \
    sentence-transformers chromadb gradio huggingface_hub requests beautifulsoup4 \
    torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
print("설치 완료 — 재시작합니다. 재시작되면 [3단계]부터 끝까지.")
os.kill(os.getpid(), 9)

## [3단계] (재시작 후) 레포 + 인증 + 벡터DB

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
assert torch.__version__.startswith("2.5.1"), "torch 2.5.1 아님 — [1단계]부터 다시"

if not os.path.exists("/content/cnu_qa_system"):
    !git clone -q https://github.com/adoveflash/cnu_qa_system.git /content/cnu_qa_system
os.chdir("/content/cnu_qa_system")
os.environ["HF_HOME"] = "/content/hf_cache"

from huggingface_hub import login, snapshot_download
login()   # gemma-3 라이선스 수락 후 토큰 붙여넣기

if not os.path.exists("data/vector_db"):
    snapshot_download(repo_id="adoveflash/cnu-qa-system", local_dir=".",
                      allow_patterns=["data/vector_db/**"])
print("torch", torch.__version__, "| 벡터DB 준비 완료")

## [4단계] Gemma 3 로드 (bfloat16 필수)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-12b-it"   # T4 OOM시 이 줄만 "google/gemma-3-4b-it"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.bfloat16)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.bfloat16)
model.eval()
print("모델 로드:", round(torch.cuda.memory_reserved()/1e9, 2), "GB | torch", torch.__version__)

## [5단계] RAG 검색 (bge-m3 + ChromaDB, top-5)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

embed_model = SentenceTransformer("BAAI/bge-m3", device="cpu")
collection = chromadb.PersistentClient(path="data/vector_db").get_collection("cnu_chunks")
print("벡터DB:", collection.count(), "청크")

def build_context(query, top_k=5):
    emb = embed_model.encode([query]).tolist()[0]
    res = collection.query(query_embeddings=[emb], n_results=top_k,
                           include=["documents", "metadatas"])
    parts, urls = [], []
    for i in range(len(res["ids"][0])):
        meta = res["metadatas"][0][i]
        parts.append(f"[참고{i+1}] {meta.get('title','')}\n{res['documents'][0][i]}")
        u = meta.get("url", "")
        if u and u not in urls:
            urls.append(u)
    return "\n\n".join(parts), urls

## [6단계] 답변 생성 — tool(실시간 식단·셔틀·일정·공지) 우선, 없으면 RAG

In [ ]:
import torch, re, gc
from datetime import datetime, timezone, timedelta
from src.tools.detector import detect_tool      # 레포에서 그대로 import (requests+bs4만 씀)
from src.tools.definitions import execute_tool

_now = datetime.now(timezone(timedelta(hours=9)))
_sem = (f"{_now.year}학년도 1학기" if 3 <= _now.month <= 8
        else f"{_now.year if _now.month >= 9 else _now.year-1}학년도 2학기")
SYSTEM = (
    f"너는 충남대학교 학내 정보를 안내하는 친절한 AI 챗봇이야.\n"
    f"오늘 날짜: {_now.strftime('%Y-%m-%d')} | 현재 학기: {_sem}\n\n"
    "- 친근하고 자연스러운 말투(~해요)로, 핵심 정보를 먼저 알려주고 필요하면 설명을 덧붙여.\n"
    "규칙:\n"
    "1. 참고 자료에 있는 정보(날짜·학점·일정 등 수치 포함)로 답하고, 구체 수치는 반드시 포함해.\n"
    "2. 참고 자료에 없는 내용은 지어내지 말고 '확인되지 않았어요'라고 해.\n"
    "3. 기숙사 식단과 학생회관 식단을 혼동하지 마. 오전/오후 시간을 정확히 구분해.\n"
    "4. 자연스러운 한국어로만 답하고, 다른 언어를 섞지 마.\n"
    "5. 졸업요건·학점은 학과마다 다르다. 묻는 학과 자료가 없으면 다른 학과 수치를 쓰지 말고 확인 안 됨이라고 해.\n"
    "6. '[참고1]' 같은 내부 표시나 '제공된 자료에 따르면' 같은 메타 표현 없이 바로 자연스럽게 답해."
)

_TOOL_FAIL = ["찾을 수 없", "가져올 수 없", "조회 실패", "정보가 없"]

def _resolve_context(question):
    """tool(실시간 식단·셔틀·학사일정·공지) 우선 시도, 실패/없으면 RAG."""
    name, args = detect_tool(question)
    if name:
        try:
            res = execute_tool(name, args)
            if res and not any(f in res for f in _TOOL_FAIL):
                print(f"  [tool] {name}")
                return f"[{name} 결과]\n{res}"
        except Exception as e:
            print(f"  [tool 실패→RAG] {e}")
    ctx, _ = build_context(question)
    return ctx

def answer(question, max_new_tokens=512):
    context = _resolve_context(question)
    user = (f"아래 참고 자료를 읽고 답해.\n\n참고 자료:\n{context}\n\n질문: {question}"
            if context else question)
    msgs = [{"role": "user", "content": [{"type": "text", "text": f"{SYSTEM}\n\n{user}"}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    n = inputs["input_ids"].shape[-1]
    try:
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, repetition_penalty=1.1)
        ans = processor.decode(out[0][n:], skip_special_tokens=True).strip()
        ans = re.sub(r"\[\s*참고\s*\d+\s*\]", "", ans).strip()
    finally:
        del inputs; gc.collect(); torch.cuda.empty_cache()
    return ans

print("RAG:", answer("컴퓨터융합학부 졸업하려면 몇 학점이야?"))
print("\nTOOL:", answer("오늘 학생회관 점심 메뉴 뭐야?"))

## [7단계] 배치 추론 → outputs/chat_output.json (tool+RAG 자동)

In [ ]:
import json, time
os.makedirs("outputs", exist_ok=True)
TEST = "data/test_chat.json"
if os.path.exists(TEST):
    data = json.load(open(TEST, encoding="utf-8"))
    results = []
    for i, item in enumerate(data):
        q = item["user"]; t = time.time()
        ans = answer(q)
        results.append({"id": item.get("id", i), "user": q, "model": ans})
        print(f"[{i+1}/{len(data)}] {q[:30]}... ({time.time()-t:.0f}s)")
    json.dump(results, open("outputs/chat_output.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)
    print("저장: outputs/chat_output.json")
else:
    print(f"{TEST} 없음 — 평가 시 조교 제공 (배치 건너뜀)")

## [8단계] Gradio UI (tool+RAG 자동)

In [ ]:
import gradio as gr

def chat_fn(message, history):
    return answer(message)

gr.ChatInterface(chat_fn, title="CNU Campus AI (Gemma 3)",
                 examples=["컴퓨터융합학부 졸업 요건", "오늘 학생회관 점심 메뉴", "셔틀버스 시간표"]
                 ).launch(share=True)